In [ ]:
"""
EEG Realtime Model Training Example on TUSZ Dataset
===================================================

This example demonstrates how to train a model for realtime seizure
detection using the Temple University Hospital (TUH) EEG Seizure Corpus
(TUSZ) dataset v2.0.5.

The model is designed to swap out models and encoders with ease.
The purpose is to achieve the best accuracy within a limited window
that is considered realtime (1 second in this case).
Two types of feature extractors, two types of models and 1 type of
evaluation method have been implemented, with CNN mainly the extract
signal features, and LSTM mainly to extract temporal features.

Note:
    Scheduler in the original code is not implemented.

Reference:
    Lee, K.; Jeong, H.; Kim, S.; Yang, D.; Kang, H.-C.; and
    Choi, E. 2022. Real-Time Seizure Detection using EEG:
    A Comprehensive Comparison of Recent Approaches under
    a Realistic Setting. arXiv e-prints, arXiv:2201.08780.
"""

'\nEEG Realtime Model Training Example on TUSZ Dataset\n===================================================\n\nThis example demonstrates how to train a model for realtime seizure\ndetection using the Temple University Hospital (TUH) EEG Seizure Corpus\n(TUSZ) dataset v2.0.5.\n\nThe model is designed to swap out models and encoders with ease.\nThe purpose is to achieve the best accuracy within a limited window\nthat is considered realtime (1 second in this case).\nTwo types of feature extractors, two types of models and 1 type of\nevaluation method have been implemented, with CNN mainly the extract\nsignal features, and LSTM mainly to extract temporal features.\n\nNote:\n    Scheduler in the original code is not implemented.\n\nReference:\n    Lee, K.; Jeong, H.; Kim, S.; Yang, D.; Kang, H.-C.; and\n    Choi, E. 2022. Real-Time Seizure Detection using EEG:\n    A Comprehensive Comparison of Recent Approaches under\n    a Realistic Setting. arXiv e-prints, arXiv:2201.08780.\n'

# Import & Install

In [ ]:
! git clone -b test-2 https://github.com/jhsieh8-stack/PyHealth.git

Cloning into 'PyHealth'...
remote: Enumerating objects: 10162, done.
remote: Counting objects: 100% (808/808), done.
remote: Compressing objects: 100% (214/214), done.
remote: Total 10162 (delta 676), reused 609 (delta 593), pack-reused 9354 (from 1)
Receiving objects: 100% (10162/10162), 133.94 MiB | 24.46 MiB/s, done.
Resolving deltas: 100% (6461/6461), done.


In [ ]:
! python --version  # Should be 3.12.x or 3.13.x

Python 3.12.13


In [ ]:
# uninstall twice just in case
! pip uninstall -y numpy scipy
! pip uninstall -y numpy scipy
! rm -rf /usr/local/lib/python3.12/dist-packages/numpy*
! cd /content/PyHealth && pip install -e . --no-cache-dir --force-reinstall

Obtaining file:///content/PyHealth
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 21.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 226.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 139.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 168.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 199.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 168.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 219.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 205.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 72.4 MB

In [ ]:
import numpy
import scipy
import torch
import torchaudio

if numpy.__version__ != '2.2.6':
  print('Force install numpy version 2.2.6. (Google colab fixed it to 2.0.2 even after installing.)')
  ! pip install numpy==2.2.6 --force-reinstall --no-deps --no-cache-dir

assert numpy.__version__ == '2.2.6'
assert scipy.__version__ == '1.17.1'
assert torch.__version__ == '2.7.1+cu126'
assert torchaudio.__version__ == '2.7.1+cu126'
assert scipy.__file__ == '/usr/local/lib/python3.12/dist-packages/scipy/__init__.py'

Force install numpy version 2.2.6. (Google colab fixed it to 2.0.2 even after installing.)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 45.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6


In [ ]:
import numpy as np
import pandas as pd
import torch
import random
import os
import math

In [ ]:
# from google.colab import drive
# drive.mount('/content/gdrive/')
# ! cp /content/gdrive/MyDrive/CS598DLH/tuh_eeg_v2.0.5.zip /content
# ! rm -rf /content/tuh_eeg_v2.0.5

Mounted at /content/gdrive/


In [ ]:
# ! cd /content && unzip -q /content/tuh_eeg_v2.0.5.zip

# assert os.path.exists('/content/tuh_eeg_v2.0.5')

# 1. Environment Setup

## Helpers

In [ ]:
def set_devices(use_cpu):
	os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
	torch.backends.cudnn.deterministic = True
	torch.backends.cudnn.benchmark = False

	if use_cpu or not torch.cuda.is_available():
		return torch.device('cpu')
	else:
		return torch.device('cuda')

In [ ]:
def set_seeds(seed):
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)
	np.random.seed(seed)
	random.seed(seed)

In [ ]:
def get_requirement_target(requirement_target, window_shift_label, window_size_label, feature_sample_rate):
  requirement_target1 = window_shift_label//2
  requirement_target2 = window_size_label//4

  if requirement_target is not None:
      requirement_target = int(requirement_target * feature_sample_rate)
  elif requirement_target1 >= requirement_target2:
      requirement_target = requirement_target1
  else:
      requirement_target = requirement_target2

  return requirement_target

In [ ]:
def get_seq_slice(x, i, window_shift_sig, window_size_sig):
    x_idx = i * window_shift_sig
    slice_start = x_idx
    slice_end = x_idx+window_size_sig
    seq_slice = x[slice_start:slice_end].permute(1, 0, 2)
    return seq_slice

In [ ]:
def get_final_target(y, i, requirement_target, window_shift_label, window_size_label):
    y_idx = i * window_shift_label
    y = y.type(torch.FloatTensor)

    slice_start = y_idx
    slice_end = y_idx+window_size_label
    target_temp = y[slice_start:slice_end]

    target, _ = torch.max(target_temp, 0)
    seiz_count = torch.count_nonzero(target_temp, dim=0)

    target[seiz_count < requirement_target] = 0

    final_target = torch.round(target).type(torch.LongTensor).squeeze()

    return final_target

## Variables

In [ ]:
%cd /content/PyHealth

/content/PyHealth


In [ ]:
! git checkout test-2 && git pull origin test-2

Already on 'test-2'
Your branch is up to date with 'origin/test-2'.
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 13 (delta 9), reused 13 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 1.33 KiB | 341.00 KiB/s, done.
From https://github.com/jhsieh8-stack/PyHealth
 * branch            test-2     -> FETCH_HEAD
   3e594f2..2831ac6  test-2     -> origin/test-2
Updating 3e594f2..2831ac6
Fast-forward
 pyhealth/datasets/tusz.py                 |   7 +-
 pyhealth/models/cnn_lstm.py               |  92 +++++++++-----
 pyhealth/models/eeg_feature_extractors.py |   2 +-
 pyhealth/models/resnet_lstm.py            | 107 +++++++++-------
 pyhealth_env/pyvenv.cfg                   |   5 +
 tests/core/test_cnn_lstm.py               | 191 +++++++++++++++++++++++++++++
 tests/core/test_resnet_lstm.py            |  84 +++++++++----
 tests/core/test_tusz.py                   |  81 +++++++

In [ ]:
SEED                = 12345
OUTPUT_DIM          = 19
BATCH_SIZE          = 32
EPOCHS              = 1 # small dataset: 30–200
LABEL_GROUP         = 'label' # label, label_bitgt_1, label_bitgt_2
DEVICE              = set_devices(use_cpu=True)
FEATURE_SAMPLE_RATE = 50
SAMPLE_RATE         = 200
WINDOW_SIZE         = 4
WINDOW_SHIFT        = 1
WINDOW_SIZE_LABEL   = FEATURE_SAMPLE_RATE*WINDOW_SIZE
WINDOW_SHIFT_LABEL  = FEATURE_SAMPLE_RATE*WINDOW_SHIFT
WINDOW_SIZE_SIG     = SAMPLE_RATE*WINDOW_SIZE
WINDOW_SHIFT_SIG    = SAMPLE_RATE*WINDOW_SHIFT
set_seeds(SEED)

# OPTIMIZER
LR_INIT       = 1e-3
WEIGHT_DECAY  = 1e-4

# SCHEDULER
LR_SCHEDULER  = "Single"
T_0 = 10
T_MULT = 1
T_UP = 1
LR_MAX = 3e-4
GAMMA = 0.5

# 2. Dataset

In [ ]:
from pyhealth.datasets import TUSZDataset

In [ ]:
train_dataset = TUSZDataset(root = '../tuh_eeg_v2.0.5', subset = 'train')
eval_dataset = TUSZDataset(root = '../tuh_eeg_v2.0.5', subset = 'eval')
test_dataset = TUSZDataset(root = '../tuh_eeg_v2.0.5', subset = 'dev')

train_dataset.stats()
eval_dataset.stats()
test_dataset.stats()

No config path provided, using default config


INFO:pyhealth.datasets.tusz:No config path provided, using default config


Wrote train metadata to ../tuh_eeg_v2.0.5/tusz-train-pyhealth.csv


INFO:pyhealth.datasets.tusz:Wrote train metadata to ../tuh_eeg_v2.0.5/tusz-train-pyhealth.csv


Initializing tusz dataset from ../tuh_eeg_v2.0.5 (dev mode: False)


...

Scanning table: dev from /content/tuh_eeg_v2.0.5/tusz-dev-pyhealth.csv


INFO:pyhealth.datasets.base_dataset:Scanning table: dev from /content/tuh_eeg_v2.0.5/tusz-dev-pyhealth.csv


Caching event dataframe to /root/.cache/pyhealth/e28e52bd-fb0c-5634-930b-c22b0e989651/global_event_df.parquet...


INFO:pyhealth.datasets.base_dataset:Caching event dataframe to /root/.cache/pyhealth/e28e52bd-fb0c-5634-930b-c22b0e989651/global_event_df.parquet...


Dataset: tusz
Dev mode: False
Number of patients: 5
Number of events: 31


In [ ]:
#train
assert len(train_dataset.unique_patient_ids) == 5
assert sum([ len(train_dataset.get_patient(patient_id).get_events()) for patient_id in train_dataset.unique_patient_ids]) == 20
assert len(train_dataset.get_patient('aaaaaaac').get_events()) == 9

# # eval
assert len(eval_dataset.unique_patient_ids) == 5
assert sum([ len(eval_dataset.get_patient(patient_id).get_events()) for patient_id in eval_dataset.unique_patient_ids]) == 73
assert len(eval_dataset.get_patient('aaaaaarq').get_events()) == 43

# # test/dev
assert len(test_dataset.unique_patient_ids) == 5
assert sum([ len(test_dataset.get_patient(patient_id).get_events()) for patient_id in test_dataset.unique_patient_ids]) == 31
assert len(test_dataset.get_patient('aaaaabxe').get_events()) == 6

Found 5 unique patient IDs


INFO:pyhealth.datasets.base_dataset:Found 5 unique patient IDs


Found 5 unique patient IDs


INFO:pyhealth.datasets.base_dataset:Found 5 unique patient IDs


Found 5 unique patient IDs


INFO:pyhealth.datasets.base_dataset:Found 5 unique patient IDs


# 3. Task

In [ ]:
from pyhealth.tasks import TUSZTask

In [ ]:
task = TUSZTask(
    sample_rate = SAMPLE_RATE,
    feature_sample_rate = FEATURE_SAMPLE_RATE,
)
train_ds = train_dataset.set_task(task)
eval_ds = train_dataset.set_task(task)
test_ds = train_dataset.set_task(task)

if len(train_ds) == 0 or len(eval_ds) == 0 or len(test_ds) == 0:
    raise RuntimeError("The task did not produce any samples.")

Setting task tusz_task for tusz base dataset...


INFO:pyhealth.datasets.base_dataset:Setting task tusz_task for tusz base dataset...


Task cache paths: task_df=/root/.cache/pyhealth/1179218f-97f9-577c-a4aa-b036ab13476a/tasks/tusz_task_b161b968-1c7d-5ccc-9b55-d331518740ad/task_df.ld, samples=/root/.cache/pyhealth/1179218f-97f9-577c-a4aa-b036ab13476a/tasks/tusz_task_b161b968-1c7d-5ccc-9b55-d331518740ad/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


INFO:pyhealth.datasets.base_dataset:Task cache paths: task_df=/root/.cache/pyhealth/1179218f-97f9-577c-a4aa-b036ab13476a/tasks/tusz_task_b161b968-1c7d-5ccc-9b55-d331518740ad/task_df.ld, samples=/root/.cache/pyhealth/1179218f-97f9-577c-a4aa-b036ab13476a/tasks/tusz_task_b161b968-1c7d-5ccc-9b55-d331518740ad/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


Applying task transformations on data with 1 workers...


...

INFO:pyhealth.datasets.base_dataset:Setting task tusz_task for tusz base dataset...


Task cache paths: task_df=/root/.cache/pyhealth/1179218f-97f9-577c-a4aa-b036ab13476a/tasks/tusz_task_b161b968-1c7d-5ccc-9b55-d331518740ad/task_df.ld, samples=/root/.cache/pyhealth/1179218f-97f9-577c-a4aa-b036ab13476a/tasks/tusz_task_b161b968-1c7d-5ccc-9b55-d331518740ad/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


INFO:pyhealth.datasets.base_dataset:Task cache paths: task_df=/root/.cache/pyhealth/1179218f-97f9-577c-a4aa-b036ab13476a/tasks/tusz_task_b161b968-1c7d-5ccc-9b55-d331518740ad/task_df.ld, samples=/root/.cache/pyhealth/1179218f-97f9-577c-a4aa-b036ab13476a/tasks/tusz_task_b161b968-1c7d-5ccc-9b55-d331518740ad/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


Found cached processed samples at /root/.cache/pyhealth/1179218f-97f9-577c-a4aa-b036ab13476a/tasks/tusz_task_b161b968-1c7d-5ccc-9b55-d331518740ad/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.


INFO:pyhealth.datasets.base_dataset:Found cached processed samples at /root/.cache/pyhealth/1179218f-97f9-577c-a4aa-b036ab13476a/tasks/tusz_task_b161b968-1c7d-5ccc-9b55-d331518740ad/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.


# 4. DataLoader

In [ ]:
from pyhealth.datasets import TUSZSamplerDataset
from torch.utils.data import DataLoader

In [ ]:
def eeg_collate_fn(samples):
    batch = []
    for sample in samples:
        signals = sample['signal']
        labels = sample[LABEL_GROUP]
        batch.append((signals, labels))

    batch_size = len(batch)
    max_seq_len = max(s[0].shape[1] for s in batch) # 6000
    num_channels = batch[0][0].shape[0]             # 20
    max_label_len = max(len(s[1]) for s in batch)

    seqs = torch.zeros(batch_size, max_seq_len, num_channels)
    targets = torch.zeros(batch_size, max_label_len, dtype=torch.long)

    seq_lengths = []
    target_lengths = []
    for i, (signals, labels) in enumerate(batch):
        seq_len = signals.shape[1]

        signals = signals.permute(1, 0)

        seqs[i, :seq_len] = signals
        targets[i, :len(labels)] = torch.tensor(labels)

        seq_lengths.append(seq_len)
        target_lengths.append(len(labels))

    return seqs, targets, seq_lengths, target_lengths

In [ ]:
# train dataset has sampler
train_sampler_ds = TUSZSamplerDataset(
    dataset=train_ds,
    is_training_set=True,
    buffer_size=32
)
train_loader = DataLoader(
    train_sampler_ds,
    batch_size=BATCH_SIZE,
    num_workers=1,
    collate_fn=eeg_collate_fn,
    drop_last=True,
    shuffle = False,
)

# eval and test datasets have no sampler
eval_loader = DataLoader(
    eval_ds,
    batch_size=BATCH_SIZE,
    num_workers=1,
    collate_fn=eeg_collate_fn,
    drop_last=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    num_workers=1,
    collate_fn=eeg_collate_fn,
    drop_last=True,
)


In [ ]:
def describe(value):
    if hasattr(value, 'shape'):
        return f"{type(value).__name__}(shape={tuple(value.shape)})"
    if isinstance(value, (list, tuple)):
        return f"{type(value).__name__}(len={len(value)})"
    return type(value).__name__

def print_summary(x, y, seq_lengths, target_lengths):
    print('Batch structure:')
    print(f"  x: {describe(x)}")
    print(f"  y: {describe(y)}")
    print(f"  seq_lengths: {describe(seq_lengths)}")
    print(f"  target_lengths: {describe(target_lengths)}")

In [ ]:
first_train_batch = next(iter(train_loader))
x, y, seq_lengths, target_lengths = first_train_batch
print_summary(x, y, seq_lengths, target_lengths)

first_eval_batch = next(iter(eval_loader))
x, y, seq_lengths, target_lengths = first_eval_batch
print_summary(x, y, seq_lengths, target_lengths)

first_train_batch = next(iter(train_loader))
x, y, seq_lengths, target_lengths = first_train_batch
print_summary(x, y, seq_lengths, target_lengths)

No control on sampler rate


/tmp/ipykernel_18097/3776044698.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  targets[i, :len(labels)] = torch.tensor(labels)


No control on sampler rate


Batch structure:
  x: Tensor(shape=(32, 6000, 20))
  y: Tensor(shape=(32, 1500))
  seq_lengths: list(len=32)
  target_lengths: list(len=32)


/tmp/ipykernel_18097/3776044698.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  targets[i, :len(labels)] = torch.tensor(labels)


Batch structure:
  x: Tensor(shape=(32, 6000, 20))
  y: Tensor(shape=(32, 1500))
  seq_lengths: list(len=32)
  target_lengths: list(len=32)
No control on sampler rate


/tmp/ipykernel_18097/3776044698.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  targets[i, :len(labels)] = torch.tensor(labels)


No control on sampler rate


Batch structure:
  x: Tensor(shape=(32, 6000, 20))
  y: Tensor(shape=(32, 1500))
  seq_lengths: list(len=32)
  target_lengths: list(len=32)


# 5. Model

In [ ]:
from pyhealth.models import CNNLSTM, ResNetLSTM

/content/PyHealth/pyhealth/metrics/calibration.py:122: SyntaxWarning: invalid escape sequence '\c'
  accuracy of 1. Thus, the ECE is :math:`\\frac{1}{3} \cdot 0.49 + \\frac{2}{3}\cdot 0.3=0.3633`.


In [ ]:
model = CNNLSTM(
    dataset    = train_sampler_ds,
    encoder    = 'raw',
    num_layers = 1, # 1-2 for 3-4 layers of cnn from feature extraction
    output_dim = OUTPUT_DIM,
    batch_size = BATCH_SIZE,
    dropout    = 0.5,
)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


# 6. Train

In [ ]:
import torch.nn as nn
import torch.optim as optim
# from torch.optim.lr_scheduler import CosineAnnealingWarmUpRestarts, CosineAnnealingWarmUp

In [ ]:
# criterion, optimizer, scheduler
criterion = nn.CrossEntropyLoss(reduction='none')
optimizer = optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
# one_epoch_iter_num = len(train_loader)
# if LR_SCHEDULER == "CosineAnnealing":
#     scheduler = CosineAnnealingWarmUpRestarts(
#         optimizer,
#         T_0     = T_0 * one_epoch_iter_num,
#         T_mult  = T_MULT,
#         eta_max = LR_MAX,
#         T_up    = T_UP * one_epoch_iter_num,
#         gamma   = GAMMA
#     )
# elif LR_SCHEDULER == "Single":
#     scheduler = CosineAnnealingWarmUpSingle(
#         optimizer,
#         max_lr          = LR_INIT * math.sqrt(BATCH_SIZE),
#         epochs          = EPOCHS,
#         steps_per_epoch = one_epoch_iter_num,
#         div_factor      = math.sqrt(BATCH_SIZE)
#     )


In [ ]:
requirement_target_initial_setting = None
requirement_target = get_requirement_target(requirement_target_initial_setting, WINDOW_SHIFT_LABEL, WINDOW_SIZE_LABEL, FEATURE_SAMPLE_RATE)

model.train()
iteration = 0
start_epoch = 1

for epoch in range(start_epoch, EPOCHS+1):
    epoch_losses =[]
    loss = 0

    for train_batch in train_loader:
        train_x, train_y, seq_lengths, target_lengths = train_batch
        train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
        iteration += 1

        ###################################
        # train with sliding window
        ###################################
        train_x = train_x.permute(1, 0, 2)
        train_y = train_y.permute(1, 0)
        iter_loss = []

        shift_start = 0
        shift_num = math.ceil((train_y.shape[0]-WINDOW_SIZE_LABEL)/float(WINDOW_SHIFT_LABEL))
        for i in range(shift_start, shift_num):
            seq_slice = get_seq_slice(train_x, i, WINDOW_SHIFT_SIG, WINDOW_SIZE_SIG)
            final_target = get_final_target(train_y, i, requirement_target, WINDOW_SHIFT_LABEL, WINDOW_SIZE_LABEL)

            optimizer.zero_grad()

            logits, _ = model(seq_slice)
            logits = logits.type(torch.FloatTensor)
            loss = criterion(logits, final_target)
            loss = torch.mean(loss)
            iter_loss.append(loss.item())

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()
            # scheduler.step(iteration)

No control on sampler rate


/tmp/ipykernel_18097/3776044698.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  targets[i, :len(labels)] = torch.tensor(labels)


No control on sampler rate


No control on sampler rate


No control on sampler rate


...

No control on sampler rate


No control on sampler rate


No control on sampler rate


No control on sampler rate


No control on sampler rate


# 7. Evaluation

In [ ]:
from pyhealth.metrics import eeg_margin_fn
from sklearn.metrics import confusion_matrix

In [ ]:
# variables
MARGIN_LIST = [3, 5]

# logger
val_losses   = 0
pred_results = []
ans_results  = []

# evaluator
probability_list      = []
final_target_list     = []
confusion_matrix_list = np.zeros((OUTPUT_DIM, OUTPUT_DIM))
labels_list           = [i for i in range(OUTPUT_DIM)]
y_true_multi          = []
y_pred_multi          = []
thresholds_margintest = []
tnr_for_margintest    = [0.7, 0.85]
binary_target_groups  = 1

In [ ]:
requirement_target_initial_setting = None
requirement_target = get_requirement_target(requirement_target_initial_setting, WINDOW_SHIFT_LABEL, WINDOW_SIZE_LABEL, FEATURE_SAMPLE_RATE)

model.eval()
with torch.no_grad():
  for idx, batch in enumerate(eval_loader):
    val_x, val_y, seq_lengths, target_lengths = batch
    val_x, val_y = val_x.to(DEVICE), val_y.to(DEVICE)


    ###################################
    # validate with sliding window
    ###################################
    val_x = val_x.permute(1, 0, 2)
    val_y = val_y.permute(1, 0)
    val_loss = []

    shift_num = math.ceil((val_y.shape[0]-WINDOW_SIZE_LABEL)/float(WINDOW_SHIFT_LABEL))

    shift_start = 0
    for i in range(shift_start, shift_num):
      seq_slice = get_seq_slice(val_x, i, WINDOW_SHIFT_SIG, WINDOW_SIZE_SIG)
      final_target = get_final_target(val_y, i, requirement_target, WINDOW_SHIFT_LABEL, WINDOW_SIZE_LABEL)

      logits, maps = model(seq_slice)
      logits = logits.type(torch.FloatTensor)

      proba = nn.functional.softmax(logits, dim=1)
      if BATCH_SIZE == 1:
          pred_results.append(proba[0])
          ans_results.append(final_target.item())
          final_target = final_target.unsqueeze(0)

      loss = criterion(logits, final_target)
      loss = torch.mean(loss)
      val_loss.append(loss.item())

      def add_batch(y_true, y_pred):
          global confusion_matrix_list
          y_pred_final = np.argmax(y_pred, axis=1)
          y_true_transformed = np.zeros((len(y_true), OUTPUT_DIM))
          y_true_transformed[range(len(y_true)), y_true] = 1
          y_pred_multi.append(y_pred)
          y_true_multi.append(y_true_transformed)
          confusion_matrix_list += confusion_matrix(y_true, y_pred_final, labels=labels_list)

      if binary_target_groups == 1:
          final_target[final_target != 0] = 1
          re_proba = torch.cat((proba[:,0].unsqueeze(1), torch.sum(proba[:,1:], 1).unsqueeze(1)), 1)
          add_batch(final_target.numpy(), re_proba.numpy())
      else:
          add_batch(final_target.numpy(), proba.numpy())

      # MARGIN TEST
      probability = proba[:, 1]
      probability_list.append(probability)
      final_target_list.append(final_target)

  val_losses += np.mean(val_loss)

print(f"val_losses: {val_losses}")
eeg_margin_fn(y_true_multi, y_pred_multi, tnr_for_margintest, probability_list, final_target_list, MARGIN_LIST)


/tmp/ipykernel_18097/3776044698.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  targets[i, :len(labels)] = torch.tensor(labels)


val_losses: 0.06741572543978691
Best threshold is:  0.04016105
1:  torch.Size([624, 32])
2:  torch.Size([624, 32])
Margin: 3, Threshold: 0.04016105, TPR: 0.3331, TNR: 0.9111
rise_accuarcy:0.0, fall_accuracy:0.0
1:  torch.Size([624, 32])
2:  torch.Size([624, 32])
Margin: 3, Threshold: 0.04016105, TPR: 0.3331, TNR: 0.9111
rise_accuarcy:0.0, fall_accuracy:0.0
1:  torch.Size([624, 32])
2:  torch.Size([624, 32])
Margin: 5, Threshold: 0.04016105, TPR: 0.3331, TNR: 0.9111
rise_accuarcy:0.0, fall_accuracy:0.0
1:  torch.Size([624, 32])
2:  torch.Size([624, 32])
Margin: 5, Threshold: 0.04016105, TPR: 0.3331, TNR: 0.9111
rise_accuarcy:0.0, fall_accuracy:0.0


(array([0.5574, 0.2886, 0.3565]),
 np.float64(0.3331),
 np.float64(0.6669),
 np.float64(0.9111),
 np.float64(0.0889))